# Konstruksi dan Dokumentasi Data
## Studi Kasus: Data Transaksi Ayam Serayu (2023–2025)

**Self-Practice — Konstruksi dan Dokumentasi Data**

Notebook ini mengerjakan Langkah 1–9 sesuai instruksi tugas praktikum, memakai dataset
transaksi Point-of-Sale rumah makan **Ayam Serayu** selama 3 tahun (3 outlet, 17 produk).

Dataset ini dipilih karena memenuhi syarat "berbagai jenis data":

| Jenis data | Contoh kolom |
|---|---|
| Numerik | `Jumlah Produk`, `Harga Produk`, `Penjualan Kotor`, `Total` |
| Kategorik | `Outlet`, `Tipe Penjualan`, `Kategori`, `Metode Pembayaran` |
| Teks / identifier | `ID Struk`, `Nama Produk`, `Kasir` |
| Temporal | `Tanggal & Waktu` |

**Alur notebook**

| Langkah | Isi |
|---|---|
| 1 | Analisis, telaah, dan validasi data |
| 2 | Strategi pembersihan data |
| 3 | Koreksi data kotor |
| 4 | Transformasi data & rekayasa fitur |
| 5 | Dokumentasi konstruksi data |
| 6 | Pelabelan data (audit SOP + penerapan) |
| 7 | Laporan hasil pelabelan data |
| 8 | Visualisasi data |
| 9 | Evaluasi dan dokumentasi akhir |

> **Cara menjalankan:** letakkan file `AyamSerayu_3Years_Transaction_Data.csv` di folder yang sama
> dengan notebook ini, lalu `Run All`. Semua output (CSV hasil olahan + gambar) otomatis
> tersimpan ke folder `output/`.
>
> **Dependency:** hanya `pandas`, `numpy`, dan `matplotlib`.

---
## Langkah 0 — Persiapan Lingkungan

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

NAMA_FILE = "AyamSerayu_3Years_Transaction_Data.csv"
KANDIDAT_PATH = [
    NAMA_FILE,
    os.path.join("data", NAMA_FILE),
    os.path.join("dataset", NAMA_FILE),
    os.path.join("/mnt/user-data/uploads", NAMA_FILE),
]

DATA_PATH = None
for _p in KANDIDAT_PATH:
    if os.path.exists(_p):
        DATA_PATH = _p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"'{NAMA_FILE}' tidak ditemukan. Letakkan file CSV di folder yang sama dengan "
        f"notebook ini, atau ubah variabel DATA_PATH secara manual.\n"
        f"Lokasi yang sudah dicek: {KANDIDAT_PATH}"
    )

OUTPUT_DIR = "output"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", matplotlib.__version__)
print("Sumber data:", DATA_PATH)
print("Folder hasil:", os.path.abspath(OUTPUT_DIR))

---
# LANGKAH 1 — Analisis, Telaah, dan Validasi Data

Tujuan langkah ini: memahami *apa* isi dataset sebelum menyentuh apa pun. Urutan telaah:

1. Muat data & lihat bentuknya
2. Struktur, tipe data, dan penggunaan memori
3. Missing value & duplikasi
4. Statistik deskriptif (numerik) dan kardinalitas (kategorik)
5. **Validasi aturan bisnis** — bagian paling penting, karena data ini "bersih di permukaan"
6. Deteksi outlier
7. Kolom tanpa variansi
8. Cakupan dan kelengkapan temporal

## 1.1 Memuat Data & Tinjauan Awal

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print(f"Jumlah baris  : {df_raw.shape[0]:,}")
print(f"Jumlah kolom  : {df_raw.shape[1]:,}")
print(f"Ukuran memori : {df_raw.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")

df_raw.head(10)

## 1.2 Struktur dan Tipe Data

In [ ]:
ringkasan_kolom = pd.DataFrame({
    "tipe_data": df_raw.dtypes.astype(str),
    "n_unik": df_raw.nunique(),
    "n_kosong": df_raw.isna().sum(),
    "pct_kosong": (df_raw.isna().mean() * 100).round(2),
    "contoh_nilai": [df_raw[c].dropna().iloc[0] for c in df_raw.columns],
})
ringkasan_kolom.index.name = "kolom"
ringkasan_kolom

In [ ]:
# Klasifikasi peran setiap kolom -> menentukan perlakuan di langkah berikutnya
peran_kolom = {
    "Tanggal & Waktu":   "Temporal (masih bertipe teks -> perlu parsing)",
    "ID Struk":          "Identifier transaksi",
    "Outlet":            "Kategorik nominal",
    "Tipe Penjualan":    "Kategorik nominal",
    "Kasir":             "Kategorik nominal (high-ish cardinality)",
    "Nama Produk":       "Kategorik nominal / teks",
    "Kategori":          "Kategorik nominal (label bawaan dataset)",
    "Jumlah Produk":     "Numerik diskret",
    "Harga Produk":      "Numerik kontinu (atribut produk, bukan atribut transaksi)",
    "Penjualan Kotor":   "Numerik kontinu (level baris item)",
    "Total":             "Numerik kontinu (level struk, DIREPLIKASI di tiap baris)",
    "Metode Pembayaran": "Kategorik nominal",
    "Status Pembayaran": "Kategorik nominal",
    "Diskon":            "Numerik kontinu",
    "Pajak":             "Numerik kontinu",
}
pd.DataFrame.from_dict(peran_kolom, orient="index", columns=["peran / catatan"])

**Catatan penting sejak awal:** dataset ini bergranularitas **baris item**, bukan transaksi.
Satu struk dengan 5 produk menghasilkan 5 baris, dan kolom `Total` (nilai struk) **ditulis ulang
identik di kelima baris tersebut**. Konsekuensinya, `df["Total"].sum()` akan menggelembungkan omzet
secara masif. Ini akan divalidasi di 1.5.

## 1.3 Missing Value dan Duplikasi

In [ ]:
print("=== MISSING VALUE ===")
mv = df_raw.isna().sum()
if mv.sum() == 0:
    print("Tidak ada missing value pada seluruh kolom.")
else:
    print(mv[mv > 0])

print("\n=== NILAI KOSONG TERSEMBUNYI (string kosong / spasi / placeholder) ===")
kolom_teks = df_raw.select_dtypes(include="object").columns
placeholder = {"", " ", "-", "N/A", "NA", "null", "NULL", "none", "None", "?"}
temuan = {}
for c in kolom_teks:
    n = df_raw[c].astype(str).str.strip().isin(placeholder).sum()
    if n:
        temuan[c] = n
print(temuan if temuan else "Tidak ada placeholder tersembunyi.")

print("\n=== DUPLIKASI ===")
n_dup = df_raw.duplicated().sum()
print(f"Baris duplikat penuh (semua kolom identik): {n_dup:,} "
      f"({n_dup / len(df_raw) * 100:.2f}% dari total baris)")

In [ ]:
# Duplikat penuh TIDAK langsung dihapus. Kita periksa dulu bentuknya.
contoh_dup = df_raw[df_raw.duplicated(keep=False)].sort_values(["ID Struk", "Nama Produk"])
print(f"Total baris yang terlibat dalam duplikasi: {len(contoh_dup):,}\n")
contoh_dup.head(8)

Terlihat polanya: **produk yang sama, di struk yang sama, dicatat sebagai dua baris terpisah
dengan `Jumlah Produk = 1`** (bukan satu baris dengan `Jumlah Produk = 2`). Apakah ini kesalahan
input ganda atau pencatatan yang sah? Jangan menebak — nanti diuji secara empiris di sel 1.5.4.

## 1.4 Statistik Deskriptif dan Kardinalitas

In [ ]:
kolom_numerik = ["Jumlah Produk", "Harga Produk", "Penjualan Kotor",
                 "Total", "Diskon", "Pajak"]
df_raw[kolom_numerik].describe().T

In [ ]:
kolom_kategorik = ["Outlet", "Tipe Penjualan", "Kasir", "Kategori",
                   "Metode Pembayaran", "Status Pembayaran"]

for c in kolom_kategorik:
    vc = df_raw[c].value_counts()
    print(f"--- {c} ({vc.size} nilai unik) ---")
    print(vc.head(10).to_string())
    print()

print(f"--- Nama Produk ({df_raw['Nama Produk'].nunique()} produk) ---")
print(df_raw["Nama Produk"].value_counts().to_string())

## 1.5 Validasi Aturan Bisnis (Integrity Check)

Statistik deskriptif tidak akan menemukan data kotor pada dataset ini, karena tidak ada
missing value dan tidak ada nilai di luar rentang wajar. Data kotornya bersifat **inkonsistensi
logis antar-kolom**. Empat aturan diuji berikut ini.

In [ ]:
# Aturan 1: Jumlah Produk x Harga Produk harus sama dengan Penjualan Kotor
selisih_kotor = df_raw["Jumlah Produk"] * df_raw["Harga Produk"] - df_raw["Penjualan Kotor"]
n_langgar = (selisih_kotor != 0).sum()

print("ATURAN 1 — Jumlah x Harga == Penjualan Kotor")
print(f"  Baris yang melanggar : {n_langgar:,}")
print("  Status               :", "LOLOS" if n_langgar == 0 else "GAGAL")

In [ ]:
# Aturan 2: satu produk harus punya satu harga dan satu kategori yang konsisten
konsistensi_produk = df_raw.groupby("Nama Produk").agg(
    n_harga_unik=("Harga Produk", "nunique"),
    harga=("Harga Produk", "first"),
    n_kategori_unik=("Kategori", "nunique"),
    kategori=("Kategori", "first"),
)

print("ATURAN 2 — Konsistensi atribut produk")
print(f"  Produk dengan >1 harga    : {(konsistensi_produk.n_harga_unik > 1).sum()}")
print(f"  Produk dengan >1 kategori : {(konsistensi_produk.n_kategori_unik > 1).sum()}")
print("  Status                    :",
      "LOLOS" if (konsistensi_produk[["n_harga_unik", "n_kategori_unik"]].max().max() == 1)
      else "GAGAL")
print()
konsistensi_produk[["harga", "kategori"]].sort_values("harga", ascending=False)

In [ ]:
# Aturan 3: ID Struk seharusnya mengidentifikasi satu transaksi secara unik
cek_struk = df_raw.groupby("ID Struk").agg(
    n_waktu=("Tanggal & Waktu", "nunique"),
    n_outlet=("Outlet", "nunique"),
    n_kasir=("Kasir", "nunique"),
    n_total=("Total", "nunique"),
)

print("ATURAN 3 — Keunikan ID Struk")
print(f"  ID Struk unik                   : {df_raw['ID Struk'].nunique():,}")
print(f"  ID Struk dengan >1 timestamp    : {(cek_struk.n_waktu   > 1).sum():,}")
print(f"  ID Struk dengan >1 outlet       : {(cek_struk.n_outlet  > 1).sum():,}")
print(f"  ID Struk dengan >1 nilai Total  : {(cek_struk.n_total   > 1).sum():,}")
print("  Status                          : GAGAL — terjadi tabrakan ID (ID collision)")

id_tabrakan = cek_struk[cek_struk.n_waktu > 1].index[0]
print(f"\nContoh tabrakan pada ID '{id_tabrakan}':")
df_raw[df_raw["ID Struk"] == id_tabrakan].sort_values("Tanggal & Waktu")

**Diagnosis Aturan 3.** Format ID adalah `TRX-<YYYYMMDD>-<4 digit>`. Sufiks 4 digit hanya
menyediakan 10.000 kemungkinan per hari, sementara volume transaksi harian melebihi angka itu,
sehingga **ID Struk berulang untuk transaksi yang berbeda di hari yang sama**.

Implikasinya serius: `groupby("ID Struk")` akan **menggabungkan dua transaksi berbeda** menjadi satu.
Solusinya adalah membentuk **kunci komposit** dari `ID Struk` + `Tanggal & Waktu` + `Outlet`.
Kita uji apakah kunci komposit ini benar-benar memulihkan integritas.

In [ ]:
# Aturan 4: sum(Penjualan Kotor) per transaksi harus sama dengan Total
# Dibandingkan antara kunci naif (ID Struk) vs kunci komposit.

def uji_rekonsiliasi(frame, kunci, nama_kunci):
    g = frame.groupby(kunci).agg(
        sum_kotor=("Penjualan Kotor", "sum"),
        total=("Total", "first"),
        n_total_unik=("Total", "nunique"),
    )
    n_beda = int((g.sum_kotor != g.total).sum())
    print(f"  [{nama_kunci}]")
    print(f"     Transaksi terdeteksi        : {len(g):,}")
    print(f"     Total tidak konsisten       : {int((g.n_total_unik > 1).sum()):,}")
    print(f"     sum(Kotor) != Total         : {n_beda:,}")
    print(f"     Status                      : {'LOLOS' if n_beda == 0 else 'GAGAL'}")
    return g

kunci_komposit = (df_raw["ID Struk"] + " | " + df_raw["Tanggal & Waktu"]
                  + " | " + df_raw["Outlet"])

print("ATURAN 4 — Rekonsiliasi nilai transaksi\n")
_ = uji_rekonsiliasi(df_raw, "ID Struk", "Kunci naif: ID Struk")
print()
g_komposit = uji_rekonsiliasi(df_raw, kunci_komposit, "Kunci komposit: ID Struk + Waktu + Outlet")

**Hasil kunci.** Dengan kunci komposit, **100% transaksi rekonsiliasi sempurna**
(`sum(Penjualan Kotor) == Total`). Artinya:

- Data transaksinya sendiri **tidak korup**.
- "2.416 struk bermasalah" yang terdeteksi kunci naif adalah **artefak dari tabrakan ID**, bukan
  data kotor sungguhan.

Ini contoh konkret kenapa validasi harus dilakukan sebelum pembersihan: kalau langsung "membersihkan"
2.416 struk tadi, kita akan membuang data yang sebenarnya sehat.

In [ ]:
# 1.5.4 — Menguji sifat baris duplikat menggunakan Total sebagai "kunci jawaban".
# Logika uji: kalau baris duplikat itu kesalahan input ganda, MENGHAPUSNYA seharusnya
# membuat sum(Penjualan Kotor) makin cocok dengan Total. Kalau justru merusak
# kecocokan, berarti baris tersebut SAH.

df_tmp = df_raw.copy()
df_tmp["_kunci"] = kunci_komposit

kunci_ber_duplikat = set(df_tmp.loc[df_tmp.duplicated(subset=df_raw.columns.tolist(),
                                                      keep=False), "_kunci"])

df_tanpa_dup = df_tmp.drop_duplicates(subset=df_raw.columns.tolist())
g_tanpa_dup = df_tanpa_dup.groupby("_kunci").agg(
    sum_kotor=("Penjualan Kotor", "sum"), total=("Total", "first"))
rusak = g_tanpa_dup.loc[list(kunci_ber_duplikat)]
n_rusak = int((rusak.sum_kotor != rusak.total).sum())

print("UJI SIFAT DUPLIKAT")
print(f"  Transaksi yang mengandung baris duplikat : {len(kunci_ber_duplikat):,}")
print(f"  Sebelum dihapus, sum(Kotor) != Total     : 0")
print(f"  Setelah  dihapus, sum(Kotor) != Total    : {n_rusak:,}")
print()
print("  KESIMPULAN: menghapus baris duplikat MERUSAK rekonsiliasi pada "
      f"{n_rusak / max(len(kunci_ber_duplikat), 1) * 100:.1f}% transaksi terdampak.")
print("  -> Baris duplikat adalah pencatatan SAH (item sama dipesan pada baris terpisah),")
print("     BUKAN kesalahan input ganda. Baris ini TIDAK BOLEH dihapus.")

del df_tmp, df_tanpa_dup, g_tanpa_dup

In [ ]:
# 1.5.5 — Anomali level atribut: metode pembayaran berbeda dalam satu transaksi
df_cek = df_raw.copy()
df_cek["_kunci"] = kunci_komposit
n_metode = df_cek.groupby("_kunci")["Metode Pembayaran"].nunique()

print("ANOMALI — Metode pembayaran tidak seragam dalam satu transaksi")
print(f"  Total transaksi                      : {len(n_metode):,}")
print(f"  Transaksi dengan >1 metode pembayaran: {(n_metode > 1).sum():,} "
      f"({(n_metode > 1).mean() * 100:.1f}%)")
print()
print("Contoh:")
kunci_contoh = n_metode[n_metode > 1].index[0]
display(df_cek[df_cek["_kunci"] == kunci_contoh][
    ["ID Struk", "Nama Produk", "Penjualan Kotor", "Metode Pembayaran", "Total"]])
del df_cek

**Interpretasi.** Secara proses bisnis, metode pembayaran adalah atribut **transaksi**, bukan
atribut **item** — pelanggan tidak membayar rendang pakai QRIS lalu es teh pakai tunai dalam satu
struk yang sama. Sekitar 76% transaksi melanggar ini, yang menandakan kolom `Metode Pembayaran`
kemungkinan besar **di-generate acak per baris**.

Karena proporsinya sangat besar, menghapus baris bukan opsi. Perlakuan yang tepat adalah
**menaikkan kolom ini ke level transaksi** dan mencatat keterbatasannya secara eksplisit di laporan.

## 1.6 Deteksi Outlier

In [ ]:
# Outlier level baris item
print("=== LEVEL BARIS ITEM ===")
print("Distribusi Jumlah Produk:")
print(df_raw["Jumlah Produk"].value_counts().sort_index().to_string())
print("\n-> Rentang 1-3 saja. Tidak ada outlier pada level item.\n")

# Outlier level transaksi (pakai kunci komposit)
nilai_transaksi = g_komposit["total"]
q1, q3 = nilai_transaksi.quantile([0.25, 0.75])
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier = nilai_transaksi[(nilai_transaksi < batas_bawah) | (nilai_transaksi > batas_atas)]

print("=== LEVEL TRANSAKSI (metode IQR) ===")
print(f"  Q1 / Q3          : Rp{q1:,.0f} / Rp{q3:,.0f}")
print(f"  IQR              : Rp{iqr:,.0f}")
print(f"  Batas bawah/atas : Rp{batas_bawah:,.0f} / Rp{batas_atas:,.0f}")
print(f"  Jumlah outlier   : {len(outlier):,} ({len(outlier)/len(nilai_transaksi)*100:.2f}%)")
print(f"  Nilai maksimum   : Rp{nilai_transaksi.max():,.0f}")

**Keputusan outlier:** outlier di sini adalah transaksi bernilai besar yang **sah secara bisnis**
(pesanan rombongan / keluarga besar). Nilai maksimumnya masih masuk akal untuk satu struk rumah makan,
dan semuanya lolos uji rekonsiliasi di Aturan 4. Maka outlier **dipertahankan**, tidak di-trim maupun
di-winsorize. Yang dilakukan hanyalah menyediakan versi **log-transform** di Langkah 4 agar distribusi
yang miring kanan tidak mendominasi model berbasis jarak.

## 1.7 Kolom Tanpa Variansi

In [ ]:
variansi_kolom = pd.DataFrame({
    "n_unik": df_raw.nunique(),
    "nilai_tunggal": [df_raw[c].iloc[0] if df_raw[c].nunique() == 1 else "-"
                      for c in df_raw.columns],
})
tanpa_variansi = variansi_kolom[variansi_kolom.n_unik == 1]

print("Kolom dengan variansi nol (hanya 1 nilai unik):")
print(tanpa_variansi.to_string() if len(tanpa_variansi) else "Tidak ada.")
print("\n-> Kolom ini tidak memiliki daya pembeda untuk analisis maupun pemodelan.")

## 1.8 Cakupan dan Kelengkapan Temporal

In [ ]:
waktu = pd.to_datetime(df_raw["Tanggal & Waktu"], errors="coerce")

print(f"Timestamp gagal di-parse : {waktu.isna().sum():,}")
print(f"Rentang data             : {waktu.min()}  s/d  {waktu.max()}")
print(f"Durasi                   : {(waktu.max() - waktu.min()).days:,} hari")

hari_ada = waktu.dt.normalize().nunique()
hari_seharusnya = (waktu.max().normalize() - waktu.min().normalize()).days + 1
print(f"Hari dengan transaksi    : {hari_ada:,} dari {hari_seharusnya:,} hari kalender")
print(f"Hari tanpa transaksi     : {hari_seharusnya - hari_ada:,}")

print("\nDistribusi transaksi per jam:")
print(waktu.dt.hour.value_counts().sort_index().to_string())

---
## Ringkasan Temuan Langkah 1

| # | Temuan | Tingkat keparahan | Bukti |
|---|---|---|---|
| T1 | Granularitas baris item, tapi `Total` direplikasi per baris | Tinggi — risiko *double counting* | Sel 1.5 Aturan 4 |
| T2 | `ID Struk` tidak unik (tabrakan ID) | Tinggi — merusak agregasi | 2.416 ID punya >1 timestamp |
| T3 | Metode pembayaran tidak seragam dalam satu transaksi | Sedang — anomali logis | ~76% transaksi |
| T4 | 10.416 baris duplikat penuh, tetapi **sah** | Rendah — jangan dihapus | Uji rekonsiliasi 1.5.4 |
| T5 | `Status Pembayaran`, `Diskon`, `Pajak` bervariansi nol | Rendah — kandidat dibuang | Sel 1.7 |
| T6 | `Tanggal & Waktu` masih bertipe teks | Rendah — perlu parsing | Sel 1.2 |
| T7 | Outlier transaksi bernilai besar, sah secara bisnis | Rendah — dipertahankan | Sel 1.6 |
| T8 | Tidak ada missing value sama sekali | — | Sel 1.3 |

**Poin utama:** dataset ini **tidak memiliki missing value**, sehingga teknik imputasi
(mean/median/mode) yang lazim dicontohkan **tidak relevan** di sini. Masalah kualitas datanya
bersifat *struktural dan logis*, dan itulah yang ditangani di Langkah 2 dan 3.

---
# LANGKAH 2 — Strategi Pembersihan Data

Setiap temuan Langkah 1 dipetakan ke satu keputusan beserta alasannya. Prinsip yang dipegang:
**tidak menghapus data kecuali terbukti salah**, dan setiap keputusan harus bisa dipertahankan
saat ditanya.

In [ ]:
strategi = pd.DataFrame([
    {
        "kode": "T1", "temuan": "Kolom Total direplikasi di tiap baris item",
        "strategi": "Pertahankan kolom (ganti nama jadi 'total_struk'); bentuk tabel agregat level transaksi di Langkah 4",
        "alasan": "Nilainya benar, hanya salah level. Menghapus akan menghilangkan kunci rekonsiliasi.",
        "risiko_bila_diabaikan": "Omzet terhitung berlipat ganda",
    },
    {
        "kode": "T2", "temuan": "ID Struk tidak unik (tabrakan ID)",
        "strategi": "Bentuk 'id_transaksi' = ID Struk + timestamp + outlet",
        "alasan": "Kunci komposit terbukti membuat 100% transaksi rekonsiliasi (Aturan 4).",
        "risiko_bila_diabaikan": "Dua transaksi berbeda tergabung jadi satu",
    },
    {
        "kode": "T3", "temuan": "Metode pembayaran beragam dalam satu transaksi",
        "strategi": "Naikkan ke level transaksi memakai MODUS per transaksi; simpan kolom asli",
        "alasan": "Terdampak ~76% transaksi sehingga penghapusan mustahil. Modus deterministik dan reversibel.",
        "risiko_bila_diabaikan": "Analisis metode pembayaran bias & tidak masuk akal secara bisnis",
    },
    {
        "kode": "T4", "temuan": "10.416 baris duplikat penuh",
        "strategi": "PERTAHANKAN (tidak dihapus)",
        "alasan": "Uji empiris 1.5.4: penghapusan justru merusak rekonsiliasi pada 100% transaksi terdampak.",
        "risiko_bila_diabaikan": "Kuantitas & omzet berkurang keliru bila dihapus",
    },
    {
        "kode": "T5", "temuan": "Status Pembayaran / Diskon / Pajak bervariansi nol",
        "strategi": "Buang dari dataset kerja; catat di dokumentasi",
        "alasan": "Tidak punya daya pembeda; hanya menambah beban memori dan dimensi.",
        "risiko_bila_diabaikan": "Noise dimensi; ilusi 'fitur tersedia' padahal konstan",
    },
    {
        "kode": "T6", "temuan": "Tanggal & Waktu bertipe teks",
        "strategi": "Parsing ke datetime64",
        "alasan": "Prasyarat semua rekayasa fitur temporal di Langkah 4.",
        "risiko_bila_diabaikan": "Pengurutan & ekstraksi komponen waktu gagal/keliru",
    },
    {
        "kode": "T7", "temuan": "Outlier nilai transaksi besar",
        "strategi": "Pertahankan; sediakan versi log-transform",
        "alasan": "Sah secara bisnis (pesanan rombongan) dan lolos rekonsiliasi.",
        "risiko_bila_diabaikan": "Kehilangan segmen pelanggan bernilai tinggi",
    },
    {
        "kode": "-", "temuan": "Nama kolom mengandung spasi & simbol '&'",
        "strategi": "Standardisasi ke snake_case",
        "alasan": "Mencegah bug akses kolom dan menyeragamkan gaya penulisan kode.",
        "risiko_bila_diabaikan": "Error sintaks / typo saat pemrosesan lanjutan",
    },
    {
        "kode": "-", "temuan": "Missing value",
        "strategi": "TIDAK ADA tindakan imputasi",
        "alasan": "Tidak ditemukan satu pun missing value maupun placeholder tersembunyi.",
        "risiko_bila_diabaikan": "-",
    },
])
strategi.set_index("kode")

---
# LANGKAH 3 — Koreksi Data Kotor

Eksekusi strategi Langkah 2, dengan **sel verifikasi** setelahnya untuk memastikan tidak ada
data yang bocor atau berubah tanpa sengaja.

## 3.1 Snapshot Kondisi Awal (untuk perbandingan sebelum–sesudah)

In [ ]:
snapshot_awal = {
    "jumlah_baris": len(df_raw),
    "jumlah_kolom": df_raw.shape[1],
    "total_kuantitas": int(df_raw["Jumlah Produk"].sum()),
    "total_penjualan_kotor": int(df_raw["Penjualan Kotor"].sum()),
    "jumlah_produk_unik": df_raw["Nama Produk"].nunique(),
}
for k, v in snapshot_awal.items():
    print(f"{k:<24}: {v:,}")

## 3.2 Standardisasi Nama Kolom

In [ ]:
PETA_KOLOM = {
    "Tanggal & Waktu":   "waktu",
    "ID Struk":          "id_struk",
    "Outlet":            "outlet",
    "Tipe Penjualan":    "tipe_penjualan",
    "Kasir":             "kasir",
    "Nama Produk":       "nama_produk",
    "Kategori":          "kategori",
    "Jumlah Produk":     "jumlah_produk",
    "Harga Produk":      "harga_produk",
    "Penjualan Kotor":   "penjualan_kotor",
    "Total":             "total_struk",
    "Metode Pembayaran": "metode_pembayaran_baris",
    "Status Pembayaran": "status_pembayaran",
    "Diskon":            "diskon",
    "Pajak":             "pajak",
}

df = df_raw.rename(columns=PETA_KOLOM).copy()
print("Nama kolom setelah standardisasi:")
print(list(df.columns))

## 3.3 Parsing Tipe Data (T6)

In [ ]:
df["waktu"] = pd.to_datetime(df["waktu"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Kolom kategorik -> tipe category (hemat memori & eksplisit secara semantik)
for c in ["outlet", "tipe_penjualan", "kasir", "nama_produk",
          "kategori", "metode_pembayaran_baris"]:
    df[c] = df[c].astype("category")

print(f"Timestamp gagal parse : {df['waktu'].isna().sum():,}")
print(f"Memori sebelum        : {df_raw.memory_usage(deep=True).sum()/1024**2:,.1f} MB")
print(f"Memori sesudah        : {df.memory_usage(deep=True).sum()/1024**2:,.1f} MB")
df.dtypes

## 3.4 Membentuk Kunci Transaksi yang Benar (T2)

In [ ]:
df["id_transaksi"] = (
    df["id_struk"].astype(str) + "|"
    + df["waktu"].dt.strftime("%Y%m%d%H%M%S") + "|"
    + df["outlet"].astype(str)
)

n_naif = df["id_struk"].nunique()
n_benar = df["id_transaksi"].nunique()

print(f"Transaksi menurut ID Struk (naif)  : {n_naif:,}")
print(f"Transaksi menurut kunci komposit   : {n_benar:,}")
print(f"Transaksi yang tadinya 'hilang'    : {n_benar - n_naif:,}")

# Verifikasi ulang rekonsiliasi dengan kunci baru
cek = df.groupby("id_transaksi", observed=True).agg(
    sum_kotor=("penjualan_kotor", "sum"), total=("total_struk", "first"))
print(f"\nTransaksi dengan sum(kotor) != total: {int((cek.sum_kotor != cek.total).sum()):,}")
print("Status: " + ("LOLOS" if (cek.sum_kotor == cek.total).all() else "GAGAL"))

## 3.5 Menormalkan Metode Pembayaran ke Level Transaksi (T3)

In [ ]:
# Modus per transaksi, dihitung secara vektor (tie-break alfabetis agar deterministik)
hitung = (df.groupby(["id_transaksi", "metode_pembayaran_baris"], observed=True)
            .size().reset_index(name="n"))
hitung = hitung.sort_values(["id_transaksi", "n", "metode_pembayaran_baris"],
                            ascending=[True, False, True])
modus_metode = hitung.drop_duplicates("id_transaksi").set_index("id_transaksi")["metode_pembayaran_baris"]

df["metode_pembayaran"] = df["id_transaksi"].map(modus_metode).astype("category")

print("Sebelum (level baris):")
print(df["metode_pembayaran_baris"].value_counts().to_string())
print("\nSesudah (level transaksi, dihitung sekali per transaksi):")
print(df.drop_duplicates("id_transaksi")["metode_pembayaran"].value_counts().to_string())

sisa = df.groupby("id_transaksi", observed=True)["metode_pembayaran"].nunique(dropna=False)
print(f"\nTransaksi yang masih punya >1 metode: {int((sisa > 1).sum()):,}")
del hitung

## 3.6 Membuang Kolom Tanpa Variansi (T5)

In [ ]:
KOLOM_DIBUANG = ["status_pembayaran", "diskon", "pajak"]

catatan_dibuang = pd.DataFrame({
    "kolom": KOLOM_DIBUANG,
    "nilai_konstan": [df_raw["Status Pembayaran"].iloc[0],
                      df_raw["Diskon"].iloc[0],
                      df_raw["Pajak"].iloc[0]],
    "n_unik": [df_raw[c].nunique() for c in ["Status Pembayaran", "Diskon", "Pajak"]],
    "alasan": ["Variansi nol; seluruh transaksi berstatus sama sehingga tidak ada daya pembeda",
               "Variansi nol; program diskon tidak terekam pada periode ini",
               "Variansi nol; komponen pajak tidak dipisahkan pada sistem POS ini"],
})
display(catatan_dibuang)

df = df.drop(columns=KOLOM_DIBUANG)
print(f"\nKolom tersisa ({df.shape[1]}): {list(df.columns)}")

> **Konsekuensi yang harus dicatat:** karena `status_pembayaran` dibuang, analisis ini
> **tidak mencakup transaksi gagal, void, maupun refund**. Karena `diskon` dan `pajak` konstan nol,
> seluruh nilai penjualan di notebook ini adalah **penjualan kotor**, bukan pendapatan bersih.

## 3.7 Verifikasi Integritas Setelah Pembersihan

In [ ]:
verifikasi = pd.DataFrame({
    "metrik": ["Jumlah baris", "Total kuantitas produk", "Total penjualan kotor",
               "Jumlah produk unik"],
    "sebelum": [snapshot_awal["jumlah_baris"],
                snapshot_awal["total_kuantitas"],
                snapshot_awal["total_penjualan_kotor"],
                snapshot_awal["jumlah_produk_unik"]],
    "sesudah": [len(df),
                int(df["jumlah_produk"].sum()),
                int(df["penjualan_kotor"].sum()),
                df["nama_produk"].nunique()],
})
verifikasi["selisih"] = verifikasi["sesudah"] - verifikasi["sebelum"]
verifikasi["status"] = np.where(verifikasi["selisih"] == 0, "AMAN", "BERUBAH — periksa!")
display(verifikasi)

assert len(df) == len(df_raw), "Jumlah baris berubah — ada data hilang!"
assert df["penjualan_kotor"].sum() == df_raw["Penjualan Kotor"].sum(), "Nilai penjualan berubah!"
print("\nSemua pemeriksaan integritas lolos. Tidak ada baris atau nilai yang hilang.")

In [ ]:
PATH_BERSIH = os.path.join(OUTPUT_DIR, "01_data_bersih_level_item.csv")
df.to_csv(PATH_BERSIH, index=False)
print(f"Tersimpan: {PATH_BERSIH}  ({os.path.getsize(PATH_BERSIH)/1024**2:,.1f} MB)")
df.head()

---
# LANGKAH 4 — Transformasi Data dan Rekayasa Fitur

Dikerjakan dalam empat bagian:

| Bagian | Isi |
|---|---|
| 4.1 | Rekayasa fitur temporal (level baris item) |
| 4.2 | Agregasi ke level transaksi — memperbaiki masalah granularitas T1 |
| 4.3 | Rekayasa fitur level transaksi |
| 4.4 | Normalisasi, scaling, dan encoding |

**Prinsip yang dipegang:** fitur harus dibuat pada **level analisis yang benar**. Fitur seperti
"jumlah item per struk" adalah properti transaksi, bukan properti item — menempelkannya ke tabel
level item akan membuat nilainya kembar di banyak baris dan merusak statistik.

## 4.1 Rekayasa Fitur Temporal (Level Baris Item)

In [ ]:
NAMA_HARI = {0: "Senin", 1: "Selasa", 2: "Rabu", 3: "Kamis",
             4: "Jumat", 5: "Sabtu", 6: "Minggu"}

BATAS_SESI = [-1, 5, 10, 14, 17, 21, 23]
LABEL_SESI = ["Dini Hari", "Pagi", "Makan Siang", "Sore", "Makan Malam", "Larut Malam"]

def tambah_fitur_waktu(frame, kolom_waktu="waktu"):
    """Menambahkan fitur turunan dari kolom timestamp."""
    t = frame[kolom_waktu]
    frame["tahun"]      = t.dt.year
    frame["bulan"]      = t.dt.month
    frame["tanggal"]    = t.dt.normalize()
    frame["hari_ke"]    = t.dt.dayofweek
    frame["nama_hari"]  = t.dt.dayofweek.map(NAMA_HARI)
    frame["is_akhir_pekan"] = (t.dt.dayofweek >= 5).astype(int)
    frame["jam"]        = t.dt.hour
    frame["sesi"]       = pd.cut(t.dt.hour, bins=BATAS_SESI, labels=LABEL_SESI)
    # Cyclical encoding: jam 23 dan jam 0 harus berdekatan, bukan berjauhan
    frame["jam_sin"]    = np.sin(2 * np.pi * t.dt.hour / 24)
    frame["jam_cos"]    = np.cos(2 * np.pi * t.dt.hour / 24)
    return frame

df = tambah_fitur_waktu(df)

# Fitur turunan level produk
df["tingkat_harga"] = pd.cut(
    df["harga_produk"],
    bins=[0, 9_999, 19_999, 100_000],
    labels=["Murah (<10rb)", "Menengah (10-20rb)", "Mahal (>20rb)"],
)

print("Fitur baru level item:")
print(["tahun", "bulan", "tanggal", "hari_ke", "nama_hari", "is_akhir_pekan",
       "jam", "sesi", "jam_sin", "jam_cos", "tingkat_harga"])
df[["waktu", "nama_hari", "is_akhir_pekan", "jam", "sesi",
    "jam_sin", "jam_cos", "harga_produk", "tingkat_harga"]].head()

**Kenapa cyclical encoding untuk jam?** Kalau jam diperlakukan sebagai angka biasa, jarak antara
jam 23 dan jam 0 terbaca 23 satuan, padahal nyatanya hanya 1 jam. Kalau di-one-hot, jadi 24 kolom
dan relasi urutan waktunya hilang total. Transformasi `sin`/`cos` memetakan jam ke lingkaran, sehingga
jam 23 dan jam 0 otomatis berdekatan.

In [ ]:
# Bukti singkat bahwa cyclical encoding bekerja: jarak Euclidean antar-jam
def jarak(j1, j2):
    a = np.array([np.sin(2*np.pi*j1/24), np.cos(2*np.pi*j1/24)])
    b = np.array([np.sin(2*np.pi*j2/24), np.cos(2*np.pi*j2/24)])
    return np.linalg.norm(a - b)

print("Pasangan jam   | selisih angka mentah | jarak setelah cyclical encoding")
for j1, j2 in [(23, 0), (23, 22), (12, 0), (11, 12)]:
    print(f"  {j1:>2} vs {j2:<2}      |        {abs(j1-j2):>2}            |  {jarak(j1, j2):.4f}")
print("\n-> Jam 23 vs 0 kini sejauh jam 23 vs 22. Persis seperti yang kita mau.")

## 4.2 Agregasi ke Level Transaksi (menyelesaikan T1)

In [ ]:
# Pisahkan kontribusi makanan vs minuman sebelum agregasi
df["nilai_makanan"] = np.where(df["kategori"] == "Makanan", df["penjualan_kotor"], 0)
df["nilai_minuman"] = np.where(df["kategori"] == "Minuman", df["penjualan_kotor"], 0)
df["qty_makanan"]   = np.where(df["kategori"] == "Makanan", df["jumlah_produk"], 0)
df["qty_minuman"]   = np.where(df["kategori"] == "Minuman", df["jumlah_produk"], 0)

df_trx = df.groupby("id_transaksi", observed=True).agg(
    waktu           = ("waktu", "first"),
    outlet          = ("outlet", "first"),
    tipe_penjualan  = ("tipe_penjualan", "first"),
    kasir           = ("kasir", "first"),
    metode_pembayaran = ("metode_pembayaran", "first"),
    n_baris_item    = ("nama_produk", "size"),
    n_produk_unik   = ("nama_produk", "nunique"),
    total_kuantitas = ("jumlah_produk", "sum"),
    nilai_transaksi = ("penjualan_kotor", "sum"),
    total_struk     = ("total_struk", "first"),
    nilai_makanan   = ("nilai_makanan", "sum"),
    nilai_minuman   = ("nilai_minuman", "sum"),
    qty_makanan     = ("qty_makanan", "sum"),
    qty_minuman     = ("qty_minuman", "sum"),
).reset_index()

print(f"Baris level item     : {len(df):,}")
print(f"Transaksi level struk: {len(df_trx):,}")
print(f"Rata-rata item/trx   : {len(df) / len(df_trx):.2f}")

# Verifikasi wajib
assert (df_trx["nilai_transaksi"] == df_trx["total_struk"]).all(), "Rekonsiliasi gagal!"
assert df_trx["nilai_transaksi"].sum() == df["penjualan_kotor"].sum(), "Omzet berubah!"
print("\nVerifikasi: nilai_transaksi == total_struk untuk 100% transaksi. LOLOS.")

In [ ]:
# Demonstrasi konkret bahaya double counting
omzet_salah  = df["total_struk"].sum()
omzet_benar  = df_trx["nilai_transaksi"].sum()

print("PERBANDINGAN PERHITUNGAN OMZET 3 TAHUN")
print(f"  Cara SALAH  df['total_struk'].sum()        : Rp{omzet_salah:,.0f}")
print(f"  Cara BENAR  agregasi level transaksi       : Rp{omzet_benar:,.0f}")
print(f"  Kelebihan hitung                            : Rp{omzet_salah - omzet_benar:,.0f} "
      f"({omzet_salah / omzet_benar:.2f}x lipat)")
print()
print(f"  Rata-rata nilai transaksi (BENAR)          : Rp{df_trx['nilai_transaksi'].mean():,.0f}")
print(f"  Rata-rata kalau pakai df['total_struk']    : Rp{df['total_struk'].mean():,.0f}  <- bias ke atas")

## 4.3 Rekayasa Fitur Level Transaksi

In [ ]:
df_trx = tambah_fitur_waktu(df_trx)

df_trx["rata_harga_per_item"] = df_trx["nilai_transaksi"] / df_trx["total_kuantitas"]
df_trx["rasio_nilai_minuman"] = df_trx["nilai_minuman"] / df_trx["nilai_transaksi"]
df_trx["punya_minuman"]       = (df_trx["qty_minuman"] > 0).astype(int)
df_trx["punya_makanan"]       = (df_trx["qty_makanan"] > 0).astype(int)
df_trx["is_pesanan_besar"]    = (df_trx["total_kuantitas"] >= 5).astype(int)
df_trx["is_jam_sibuk"]        = df_trx["jam"].isin([11, 12, 13, 18, 19]).astype(int)
df_trx["keragaman_produk"]    = df_trx["n_produk_unik"] / df_trx["total_kuantitas"]

fitur_baru = ["rata_harga_per_item", "rasio_nilai_minuman", "punya_minuman",
              "punya_makanan", "is_pesanan_besar", "is_jam_sibuk", "keragaman_produk"]

print("Statistik fitur turunan level transaksi:")
display(df_trx[fitur_baru].describe().T)
df_trx[["id_transaksi", "nilai_transaksi", "total_kuantitas"] + fitur_baru].head()

## 4.4 Normalisasi, Scaling, dan Encoding

In [ ]:
# --- Normalisasi & scaling numerik ---
x = df_trx["nilai_transaksi"].astype(float)

df_trx["nilai_minmax"] = (x - x.min()) / (x.max() - x.min())        # skala 0-1
df_trx["nilai_zscore"] = (x - x.mean()) / x.std(ddof=0)             # standardisasi
df_trx["nilai_log"]    = np.log1p(x)                                # meredam kemencengan

# Robust scaling: tahan terhadap outlier karena memakai median & IQR
q1_, q3_ = x.quantile([0.25, 0.75])
df_trx["nilai_robust"] = (x - x.median()) / (q3_ - q1_)

perbandingan_skala = pd.DataFrame({
    "asli (Rp)":  x.describe(),
    "min-max":    df_trx["nilai_minmax"].describe(),
    "z-score":    df_trx["nilai_zscore"].describe(),
    "log1p":      df_trx["nilai_log"].describe(),
    "robust":     df_trx["nilai_robust"].describe(),
})
print("Perbandingan hasil scaling:")
display(perbandingan_skala)

print(f"Kemencengan (skewness) sebelum log : {x.skew():.4f}")
print(f"Kemencengan (skewness) sesudah log : {df_trx['nilai_log'].skew():.4f}")

**Kapan pakai yang mana?**

| Teknik | Cocok untuk | Catatan |
|---|---|---|
| Min-Max | Neural network, algoritma yang butuh rentang tetap | Sangat sensitif outlier |
| Z-score | Regresi linier, PCA, SVM | Asumsi distribusi mendekati normal |
| Log1p | Data miring kanan seperti nilai transaksi | Aman untuk nol karena memakai `log(1+x)` |
| Robust | Data dengan outlier yang dipertahankan | Memakai median & IQR, bukan mean & std |

Untuk dataset ini, **log1p** dan **robust** paling masuk akal karena outlier sengaja dipertahankan
(keputusan di Langkah 1.6).

In [ ]:
# --- Encoding kategorik ---
# 1) Nominal tanpa urutan -> One-Hot Encoding
kolom_onehot = ["outlet", "tipe_penjualan", "metode_pembayaran"]
onehot = pd.get_dummies(df_trx[kolom_onehot].astype(str), prefix=kolom_onehot, dtype=int)

# 2) Ordinal (punya urutan alami) -> Ordinal Encoding manual
PETA_SESI = {"Dini Hari": 0, "Pagi": 1, "Makan Siang": 2,
             "Sore": 3, "Makan Malam": 4, "Larut Malam": 5}
df_trx["sesi_ordinal"] = df_trx["sesi"].astype(str).map(PETA_SESI)

# 3) Kardinalitas menengah (kasir, 8 nilai) -> Frequency Encoding
freq_kasir = df_trx["kasir"].value_counts(normalize=True)
df_trx["kasir_freq"] = df_trx["kasir"].map(freq_kasir).astype(float)

df_trx = pd.concat([df_trx, onehot], axis=1)

print(f"Kolom one-hot yang dihasilkan ({onehot.shape[1]}):")
print(list(onehot.columns))
print(f"\nDimensi tabel transaksi: {df_trx.shape}")
df_trx[["sesi", "sesi_ordinal", "kasir", "kasir_freq"] + list(onehot.columns)[:5]].head()

> **Kenapa `kasir` tidak di-one-hot?** Bisa saja (hanya 8 nilai), tapi frequency encoding dipilih
> agar tidak menambah 8 kolom untuk variabel yang secara hipotesis bukan pendorong utama nilai
> transaksi. `nama_produk` juga sengaja tidak di-encode pada level transaksi karena satu transaksi
> memuat banyak produk — representasi yang tepat untuknya adalah *basket matrix*, di luar cakupan
> tugas ini.

---
# LANGKAH 5 — Dokumentasi Konstruksi Data

Berisi kamus data fitur hasil konstruksi, teknik yang dipakai beserta alasannya, dan rekomendasi
dampak terhadap kualitas data.

## 5.1 Kamus Data Fitur Hasil Konstruksi

In [ ]:
kamus_data = pd.DataFrame([
    ("id_transaksi",        "Teks",     "Kunci komposit", "ID Struk + timestamp + outlet", "Mengatasi tabrakan ID Struk (T2)"),
    ("tahun",               "Integer",  "Ekstraksi",      "waktu.dt.year", "Analisis tren tahunan"),
    ("bulan",               "Integer",  "Ekstraksi",      "waktu.dt.month", "Deteksi musiman bulanan"),
    ("hari_ke",             "Integer",  "Ekstraksi",      "waktu.dt.dayofweek", "Pola mingguan"),
    ("nama_hari",           "Kategorik","Pemetaan",       "hari_ke -> nama hari", "Keterbacaan laporan"),
    ("is_akhir_pekan",      "Biner",    "Aturan",         "hari_ke >= 5", "Membedakan perilaku weekday/weekend"),
    ("jam",                 "Integer",  "Ekstraksi",      "waktu.dt.hour", "Pola harian"),
    ("sesi",                "Kategorik","Binning",        "pd.cut atas jam, 6 sesi", "Menyederhanakan 24 jam jadi konsep bisnis"),
    ("sesi_ordinal",        "Integer",  "Ordinal encoding","sesi -> 0..5", "Mempertahankan urutan waktu"),
    ("jam_sin / jam_cos",   "Float",    "Cyclical encoding","sin & cos (2*pi*jam/24)", "Menjaga kedekatan jam 23 dan jam 0"),
    ("tingkat_harga",       "Kategorik","Binning",        "pd.cut atas harga_produk", "Segmentasi produk murah/menengah/mahal"),
    ("n_baris_item",        "Integer",  "Agregasi",       "count baris per transaksi", "Ukuran kompleksitas pesanan"),
    ("n_produk_unik",       "Integer",  "Agregasi",       "nunique produk per transaksi", "Keragaman pesanan"),
    ("total_kuantitas",     "Integer",  "Agregasi",       "sum jumlah_produk", "Volume pesanan"),
    ("nilai_transaksi",     "Integer",  "Agregasi",       "sum penjualan_kotor", "Metrik nilai yang BENAR (bukan total_struk)"),
    ("nilai_makanan/minuman","Integer", "Agregasi kondisional","sum per kategori", "Analisis komposisi pesanan"),
    ("rata_harga_per_item", "Float",    "Rasio",          "nilai_transaksi / total_kuantitas", "Proksi kelas produk yang dipesan"),
    ("rasio_nilai_minuman", "Float",    "Rasio",          "nilai_minuman / nilai_transaksi", "Indikator perilaku upsell minuman"),
    ("punya_minuman",       "Biner",    "Aturan",         "qty_minuman > 0", "Analisis attachment rate"),
    ("is_pesanan_besar",    "Biner",    "Aturan",         "total_kuantitas >= 5", "Proksi pesanan rombongan"),
    ("is_jam_sibuk",        "Biner",    "Aturan",         "jam dalam {11,12,13,18,19}", "Analisis beban operasional"),
    ("keragaman_produk",    "Float",    "Rasio",          "n_produk_unik / total_kuantitas", "1.0 = semua item berbeda"),
    ("nilai_minmax",        "Float",    "Min-Max scaling","(x-min)/(max-min)", "Untuk algoritma berbasis jarak"),
    ("nilai_zscore",        "Float",    "Standardisasi",  "(x-mean)/std", "Untuk model linier & PCA"),
    ("nilai_log",           "Float",    "Log transform",  "log1p(x)", "Meredam kemencengan kanan"),
    ("nilai_robust",        "Float",    "Robust scaling", "(x-median)/IQR", "Tahan terhadap outlier"),
    ("kasir_freq",          "Float",    "Frequency encoding","proporsi kemunculan kasir", "Menghindari ledakan dimensi"),
    ("outlet_* / tipe_* / metode_*", "Biner", "One-Hot encoding", "pd.get_dummies", "Kategorik nominal tanpa urutan"),
], columns=["fitur", "tipe", "teknik", "formula / aturan", "tujuan analitis"])

pd.set_option("display.max_colwidth", 60)
kamus_data

In [ ]:
PATH_KAMUS = os.path.join(OUTPUT_DIR, "02_kamus_data_fitur.csv")
kamus_data.to_csv(PATH_KAMUS, index=False)
print("Tersimpan:", PATH_KAMUS)

## 5.2 Uraian Teknik Transformasi yang Digunakan

**a. Parsing tipe data.** `Tanggal & Waktu` diubah dari teks ke `datetime64`. Tanpa ini, seluruh
fitur temporal di 4.1 mustahil dibuat. Kolom kategorik diubah ke tipe `category`, yang menurunkan
penggunaan memori secara signifikan sekaligus menyatakan secara eksplisit bahwa kolom tersebut
bukan teks bebas.

**b. Agregasi granularitas.** Ini transformasi paling berdampak di notebook ini. Dataset asli
bergranularitas baris item; sebagian besar pertanyaan bisnis ("berapa rata-rata belanja pelanggan?",
"jam berapa transaksi paling ramai?") bergranularitas transaksi. Agregasi memakai kunci komposit
menurunkan 626.311 baris menjadi 208.730 transaksi, dan menghilangkan risiko *double counting*
yang terbukti menggelembungkan omzet hingga sekitar 3,6 kali lipat.

**c. Binning.** Jam (24 nilai) diringkas menjadi 6 sesi bisnis, dan harga produk menjadi 3 tingkat.
Binning mengurangi dimensi dan membuat hasil analisis lebih mudah dikomunikasikan, dengan biaya
kehilangan sebagian detail. Batas sesi ditentukan dari distribusi jam nyata (puncak di jam 12 dan
18–19), bukan dari asumsi.

**d. Cyclical encoding.** Diterapkan pada jam untuk mempertahankan sifat siklis waktu. Sudah
dibuktikan secara numerik di sel 4.1.

**e. Scaling.** Empat varian disediakan (min-max, z-score, log1p, robust) agar pengguna hilir bebas
memilih sesuai algoritmanya. Perlu diperhatikan: **kolom `harga_produk` sengaja tidak di-scale**,
karena ia hanya punya 17 nilai unik dan merupakan atribut produk, bukan hasil observasi transaksi —
men-standardisasi kolom seperti ini secara teknis bisa, tetapi tidak bermakna.

**f. Encoding kategorik.** Tiga strategi berbeda dipakai sesuai sifat variabel: one-hot untuk nominal
berkardinalitas rendah, ordinal untuk variabel berurutan, dan frequency untuk kardinalitas menengah.

## 5.3 Rekomendasi dan Dampak terhadap Kualitas Data

**Dampak positif**

1. **Akurasi metrik bisnis pulih.** Sebelum konstruksi, perhitungan omzet naif menghasilkan angka
   sekitar 3,6× lipat dari yang sebenarnya. Setelah agregasi, angkanya terrekonsiliasi 100%.
2. **Integritas kunci terjamin.** Kunci komposit memulihkan 2.434 transaksi yang sebelumnya
   tersembunyi akibat tabrakan ID.
3. **Dimensi analitis bertambah.** Dari 15 kolom mentah menjadi lebih dari 40 fitur siap analisis,
   tanpa satu pun baris data hilang.
4. **Reversibilitas terjaga.** Semua kolom asli dipertahankan berdampingan dengan versi
   transformasinya, sehingga setiap langkah dapat diaudit ulang.

**Keterbatasan yang harus dinyatakan**

1. **Kolom `metode_pembayaran` tidak dapat dipercaya sepenuhnya.** Sekitar 76% transaksi memiliki
   metode berbeda antar-baris, yang mengindikasikan data ini kemungkinan besar dibangkitkan acak.
   Modus per transaksi hanyalah pendekatan terbaik yang tersedia; analisis berbasis metode
   pembayaran wajib mencantumkan disclaimer ini.
2. **Tidak ada identitas pelanggan.** Akibatnya analisis retensi, CLV, maupun RFM tidak dapat
   dilakukan. Yang tersedia hanya analisis level transaksi.
3. **Tidak ada data biaya (HPP).** Semua metrik bersifat penjualan kotor; margin tidak dapat dihitung.
4. **Diskon dan pajak konstan nol**, sehingga selisih harga jual dan pendapatan bersih tidak terlihat.

**Rekomendasi perbaikan di sisi sumber data (POS)**

| Prioritas | Rekomendasi |
|---|---|
| Tinggi | Ubah format ID struk agar unik global, misal `TRX-<outlet>-<YYYYMMDD>-<sequence>` |
| Tinggi | Pindahkan `metode_pembayaran` ke tabel header transaksi, bukan tabel detail item |
| Sedang | Ekspor data dalam dua tabel terpisah (header & detail) agar granularitas tidak ambigu |
| Sedang | Rekam `status_pembayaran` yang sebenarnya, termasuk void dan refund |
| Rendah | Tambahkan identitas pelanggan (nomor HP / ID member) untuk membuka analisis retensi |

---
# LANGKAH 6 — Pelabelan Data

Dua sub-tugas sesuai instruksi:

1. **Audit label yang sudah ada** — memeriksa apakah pelabelan sejenis pada dataset sudah konsisten
   dengan SOP-nya.
2. **Pelabelan dataset baru** — menyusun SOP sendiri lalu menerapkannya.

## 6.1 Audit Label Bawaan (`kategori`) terhadap SOP

Dataset sudah punya satu label bawaan: kolom `kategori` (Makanan / Minuman). SOP implisitnya adalah
**setiap produk harus dipetakan ke tepat satu kategori secara konsisten di seluruh baris**.

In [ ]:
audit_kategori = df.groupby("nama_produk", observed=True).agg(
    n_kategori_unik=("kategori", "nunique"),
    kategori=("kategori", "first"),
    n_baris=("kategori", "size"),
).sort_values("kategori")

audit_kategori["status_sop"] = np.where(
    audit_kategori.n_kategori_unik == 1, "KONSISTEN", "MELANGGAR SOP")

n_langgar = int((audit_kategori.n_kategori_unik > 1).sum())
print("HASIL AUDIT LABEL BAWAAN 'kategori'")
print(f"  Produk diaudit            : {len(audit_kategori)}")
print(f"  Produk melanggar SOP      : {n_langgar}")
print(f"  Tingkat konsistensi       : {(1 - n_langgar/len(audit_kategori))*100:.2f}%")
print(f"  Kesimpulan                : {'LOLOS AUDIT' if n_langgar == 0 else 'PERLU PERBAIKAN'}")
print()
display(audit_kategori)

print("Distribusi label bawaan (level baris item):")
print(df["kategori"].value_counts().to_string())

**Hasil audit:** seluruh 17 produk terpetakan konsisten ke satu kategori. Contohnya, `Es Jeruk`
selalu berlabel `Minuman` di seluruh 37.188 kemunculannya, tanpa satu pun penyimpangan.
Label bawaan ini **lolos audit** dan dapat dipakai sebagai referensi mutu untuk pelabelan baru.

## 6.2 SOP Pelabelan Data Baru: `segmen_transaksi`

**Nama label:** `segmen_transaksi`
**Level penerapan:** transaksi (bukan baris item)
**Tujuan bisnis:** mengelompokkan transaksi berdasarkan nilai belanja, untuk mendukung penyusunan
strategi promosi dan penentuan target *upselling*.

**Aturan pelabelan**

| Label | Aturan | Rasional |
|---|---|---|
| `Kecil` | `nilai_transaksi <= Q1` | Pembelian personal / item tunggal |
| `Sedang` | `Q1 < nilai_transaksi <= Q3` | Mayoritas transaksi, pola makan reguler |
| `Besar` | `nilai_transaksi > Q3` | Pesanan keluarga / rombongan |

**Ketentuan pelaksanaan**

1. **Batas kelas dihitung dari kuantil pada level transaksi**, bukan level baris item. Menghitung
   kuantil di level item akan bias ke transaksi beranggota banyak item.
2. **Batas bersifat inklusif ke bawah** (`<=`) untuk menghilangkan ambiguitas nilai tepat di batas.
3. **Batas kuantil disimpan sebagai artefak**, agar data baru di masa depan dilabeli memakai batas
   yang sama, bukan kuantil barunya sendiri (mencegah *label drift*).
4. **Tidak boleh ada transaksi tanpa label.** Jumlah label harus persis sama dengan jumlah transaksi.
5. **Pencegahan kebocoran:** karena label diturunkan dari `nilai_transaksi`, seluruh kolom yang
   merupakan `nilai_transaksi` atau turunannya **wajib dikeluarkan** dari himpunan fitur apabila
   label ini dipakai sebagai target pemodelan.

In [ ]:
# Menghitung dan menyimpan batas kelas sebagai artefak
Q1_TRX = float(df_trx["nilai_transaksi"].quantile(0.25))
Q3_TRX = float(df_trx["nilai_transaksi"].quantile(0.75))

batas_label = {"Q1": Q1_TRX, "Q3": Q3_TRX,
               "sumber": "kuantil nilai_transaksi level transaksi",
               "n_transaksi_acuan": len(df_trx)}

print("BATAS KELAS (artefak SOP)")
for k, v in batas_label.items():
    print(f"  {k:<18}: {v:,.0f}" if isinstance(v, (int, float)) else f"  {k:<18}: {v}")

# Penerapan label
kondisi = [
    df_trx["nilai_transaksi"] <= Q1_TRX,
    (df_trx["nilai_transaksi"] > Q1_TRX) & (df_trx["nilai_transaksi"] <= Q3_TRX),
    df_trx["nilai_transaksi"] > Q3_TRX,
]
df_trx["segmen_transaksi"] = pd.Categorical(
    np.select(kondisi, ["Kecil", "Sedang", "Besar"], default=None),
    categories=["Kecil", "Sedang", "Besar"], ordered=True,
)
df_trx["segmen_ordinal"] = df_trx["segmen_transaksi"].cat.codes

print("\nContoh hasil pelabelan:")
df_trx[["id_transaksi", "nilai_transaksi", "total_kuantitas",
        "segmen_transaksi", "segmen_ordinal"]].head(10)

## 6.3 Label Kedua: `sesi` (berbasis aturan waktu)

Label `sesi` sudah dibuat di Langkah 4.1. Di sini ditegaskan SOP-nya agar bisa diaudit sama seperti
label lain.

| Label | Rentang jam | Rasional |
|---|---|---|
| Dini Hari | 00–05 | Volume sangat rendah |
| Pagi | 06–10 | Sarapan, volume rendah |
| Makan Siang | 11–14 | Puncak pertama (jam 12) |
| Sore | 15–17 | Lembah antar-puncak |
| Makan Malam | 18–21 | Puncak kedua (jam 18–19) |
| Larut Malam | 22–23 | Menurun menjelang tutup |

Batas ini **diturunkan dari distribusi jam yang teramati** di sel 1.8, bukan dari asumsi umum.

## 6.4 Validasi Hasil Pelabelan

In [ ]:
print("VALIDASI PELABELAN")

# 1. Kelengkapan
n_kosong = int(df_trx["segmen_transaksi"].isna().sum())
print(f"  1. Transaksi tanpa label            : {n_kosong:,}  "
      f"-> {'LOLOS' if n_kosong == 0 else 'GAGAL'}")

# 2. Kesesuaian jumlah
print(f"  2. Jumlah label == jumlah transaksi : "
      f"{df_trx['segmen_transaksi'].notna().sum():,} == {len(df_trx):,}  "
      f"-> {'LOLOS' if df_trx['segmen_transaksi'].notna().sum() == len(df_trx) else 'GAGAL'}")

# 3. Kepatuhan aturan batas (diuji ulang secara independen)
uji = df_trx.groupby("segmen_transaksi", observed=True)["nilai_transaksi"].agg(["min", "max"])
patuh = (uji.loc["Kecil", "max"] <= Q1_TRX
         and uji.loc["Sedang", "min"] > Q1_TRX
         and uji.loc["Sedang", "max"] <= Q3_TRX
         and uji.loc["Besar", "min"] > Q3_TRX)
print(f"  3. Kepatuhan batas kelas            : {'LOLOS' if patuh else 'GAGAL'}")
display(uji)

# 4. Sesi
n_sesi_kosong = int(df_trx["sesi"].isna().sum())
print(f"  4. Transaksi tanpa label sesi       : {n_sesi_kosong:,}  "
      f"-> {'LOLOS' if n_sesi_kosong == 0 else 'GAGAL'}")

In [ ]:
# 5. Pencegahan kebocoran label (label leakage)
KOLOM_BOCOR = ["nilai_transaksi", "total_struk", "nilai_makanan", "nilai_minuman",
               "nilai_minmax", "nilai_zscore", "nilai_log", "nilai_robust",
               "rata_harga_per_item", "rasio_nilai_minuman", "segmen_ordinal"]

print("PERINGATAN KEBOCORAN LABEL")
print("  Label 'segmen_transaksi' diturunkan langsung dari 'nilai_transaksi'.")
print("  Kolom berikut WAJIB dikeluarkan dari fitur bila label ini jadi target pemodelan:\n")
for c in KOLOM_BOCOR:
    print(f"    - {c}")

FITUR_AMAN = ["n_baris_item", "n_produk_unik", "total_kuantitas", "qty_makanan",
              "qty_minuman", "punya_minuman", "punya_makanan", "is_pesanan_besar",
              "keragaman_produk", "jam", "jam_sin", "jam_cos", "sesi_ordinal",
              "hari_ke", "is_akhir_pekan", "is_jam_sibuk", "bulan", "tahun",
              "kasir_freq"] + [c for c in df_trx.columns
                               if c.startswith(("outlet_", "tipe_penjualan_", "metode_pembayaran_"))]

print(f"\n  Fitur yang AMAN dipakai ({len(FITUR_AMAN)} kolom):")
print("   ", FITUR_AMAN)

---
# LANGKAH 7 — Laporan Hasil Pelabelan Data

## 7.1 Distribusi Label

In [ ]:
lap_segmen = pd.DataFrame({
    "jumlah_transaksi": df_trx["segmen_transaksi"].value_counts().sort_index(),
    "proporsi_%": (df_trx["segmen_transaksi"].value_counts(normalize=True).sort_index() * 100).round(2),
})
lap_segmen["kontribusi_omzet"] = df_trx.groupby("segmen_transaksi", observed=True)["nilai_transaksi"].sum()
lap_segmen["kontribusi_omzet_%"] = (lap_segmen["kontribusi_omzet"]
                                    / lap_segmen["kontribusi_omzet"].sum() * 100).round(2)
lap_segmen.index.name = "segmen_transaksi"

print("DISTRIBUSI LABEL 'segmen_transaksi'")
display(lap_segmen)

rasio = lap_segmen["jumlah_transaksi"].max() / lap_segmen["jumlah_transaksi"].min()
print(f"Rasio ketidakseimbangan (imbalance ratio): {rasio:.2f} : 1")
print("Interpretasi: " + ("relatif seimbang, tidak perlu resampling khusus." if rasio < 3
      else "cukup timpang, pertimbangkan class weighting bila dipakai untuk klasifikasi."))

In [ ]:
lap_sesi = pd.DataFrame({
    "jumlah_transaksi": df_trx["sesi"].value_counts().sort_index(),
    "proporsi_%": (df_trx["sesi"].value_counts(normalize=True).sort_index() * 100).round(2),
})
lap_sesi["omzet"] = df_trx.groupby("sesi", observed=True)["nilai_transaksi"].sum()
lap_sesi["rata_nilai_transaksi"] = df_trx.groupby("sesi", observed=True)["nilai_transaksi"].mean().round(0)
lap_sesi.index.name = "sesi"

print("DISTRIBUSI LABEL 'sesi'")
display(lap_sesi)

## 7.2 Statistik Deskriptif per Label

In [ ]:
stat_per_segmen = df_trx.groupby("segmen_transaksi", observed=True).agg(
    n=("nilai_transaksi", "size"),
    nilai_min=("nilai_transaksi", "min"),
    nilai_median=("nilai_transaksi", "median"),
    nilai_maks=("nilai_transaksi", "max"),
    rata_kuantitas=("total_kuantitas", "mean"),
    rata_produk_unik=("n_produk_unik", "mean"),
    pct_punya_minuman=("punya_minuman", "mean"),
    pct_akhir_pekan=("is_akhir_pekan", "mean"),
).round(2)
stat_per_segmen["pct_punya_minuman"] *= 100
stat_per_segmen["pct_akhir_pekan"] *= 100
stat_per_segmen

## 7.3 Tabulasi Silang Label dengan Dimensi Lain

In [ ]:
for dim in ["outlet", "tipe_penjualan", "sesi"]:
    tab = pd.crosstab(df_trx[dim], df_trx["segmen_transaksi"], normalize="index") * 100
    print(f"--- Proporsi segmen dalam tiap {dim} (%, per baris) ---")
    display(tab.round(2))

## 7.4 Evaluasi Proses Pelabelan dan Tantangan yang Ditemui

**Tantangan 1 — Menentukan level pelabelan yang benar.**
Godaan terbesarnya adalah melabeli di level baris item, karena itu bentuk asli datanya. Kalau
dilakukan, kuantil akan tertarik oleh transaksi beranggota banyak item dan batas kelasnya jadi keliru.
*Solusi:* pelabelan dilakukan setelah agregasi ke level transaksi, dan hal ini ditulis eksplisit
sebagai ketentuan nomor 1 dalam SOP.

**Tantangan 2 — Risiko kebocoran label.**
Label diturunkan dari `nilai_transaksi`. Kalau kolom itu tetap dipakai sebagai fitur, model apa pun
akan mencapai akurasi mendekati sempurna namun sama sekali tidak berguna.
*Solusi:* menyusun daftar `KOLOM_BOCOR` dan `FITUR_AMAN` secara eksplisit di sel 6.4.

**Tantangan 3 — Batas kelas yang tidak stabil.**
Jika data periode berikutnya dilabeli memakai kuantilnya sendiri, definisi "Besar" akan bergeser
tiap periode sehingga tren antar-periode tidak lagi dapat dibandingkan.
*Solusi:* batas Q1 dan Q3 disimpan sebagai artefak dalam variabel `batas_label` dan diekspor ke file.

**Tantangan 4 — Nilai transaksi bersifat diskret.**
Karena hanya ada 17 produk dengan harga tetap, nilai transaksi membentuk himpunan diskret dan banyak
transaksi bernilai persis sama. Akibatnya kuantil tidak membelah data tepat 25%/50%/25%.
*Solusi:* menerima ketimpangan ringan ini dan melaporkan rasio ketidakseimbangannya secara terbuka
di sel 7.1, alih-alih memaksa proporsi seimbang dengan cara yang mendistorsi makna label.

**Tantangan 5 — Tidak ada label kebenaran (ground truth) pembanding.**
Berbeda dengan label `kategori` yang bisa diverifikasi terhadap menu, `segmen_transaksi` adalah
konstruksi analitik tanpa acuan eksternal.
*Solusi:* validasi difokuskan pada **kepatuhan aturan** (sel 6.4) dan **kelayakan bisnis** hasilnya
(sel 7.2), bukan pada akurasi terhadap kebenaran mutlak.

**Langkah perbaikan yang diusulkan**

1. Validasi ambang batas segmen bersama tim operasional outlet, agar definisi "Besar" selaras dengan
   pemahaman bisnis, bukan semata-mata kuantil statistik.
2. Terapkan batas yang tersimpan pada data periode berikutnya, lalu pantau pergeseran distribusinya
   sebagai indikator perubahan perilaku pelanggan.
3. Kembangkan label turunan berbasis komposisi pesanan (misal "Makanan Saja" vs "Paket Lengkap") yang
   tidak diturunkan dari nilai transaksi, sehingga bebas dari masalah kebocoran.

---
# LANGKAH 8 — Visualisasi Data

Seluruh visualisasi memakai data **hasil pembersihan dan rekayasa fitur**, dan seluruh metrik nilai
dihitung pada **level transaksi** agar bebas dari double counting.

In [ ]:
WARNA = {"Kecil": "#8ecae6", "Sedang": "#219ebc", "Besar": "#023047"}

def simpan(nama):
    path = os.path.join(FIG_DIR, nama)
    plt.savefig(path, bbox_inches="tight", dpi=120)
    print("Tersimpan:", path)

## 8.1 Histogram — Distribusi Nilai Transaksi (Sebelum vs Sesudah Log)

Perhatikan angka skewness yang tercetak di bawah grafik: nilainya bergerak dari sekitar +1,08
menjadi sekitar −0,74. Nilai mutlaknya turun — distribusi jadi lebih simetris — tetapi tandanya
berbalik, artinya log1p sedikit **terlalu mengoreksi** dan memanjangkan ekor kiri. Ini wajar pada
data yang batas bawahnya terpotong tajam (transaksi minimum Rp5.000). Untuk kasus ini,
`nilai_robust` sering kali lebih aman dipakai daripada `nilai_log`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].hist(df_trx["nilai_transaksi"], bins=60, color="#219ebc", edgecolor="white")
ax[0].axvline(Q1_TRX, color="#fb8500", ls="--", lw=2, label=f"Q1 = Rp{Q1_TRX:,.0f}")
ax[0].axvline(Q3_TRX, color="#d00000", ls="--", lw=2, label=f"Q3 = Rp{Q3_TRX:,.0f}")
ax[0].set_title("Distribusi Nilai Transaksi (skala asli)")
ax[0].set_xlabel("Nilai transaksi (Rp)")
ax[0].set_ylabel("Jumlah transaksi")
ax[0].legend()

ax[1].hist(df_trx["nilai_log"], bins=60, color="#023047", edgecolor="white")
ax[1].set_title("Setelah transformasi log1p")
ax[1].set_xlabel("log(1 + nilai transaksi)")
ax[1].set_ylabel("Jumlah transaksi")

fig.suptitle("Efek Log-Transform terhadap Kemencengan Distribusi", fontsize=13, y=1.02)
plt.tight_layout()
simpan("01_histogram_nilai_transaksi.png")
plt.show()

print(f"Skewness sebelum log : {df_trx['nilai_transaksi'].skew():.4f}")
print(f"Skewness sesudah log : {df_trx['nilai_log'].skew():.4f}")

## 8.2 Box Plot — Nilai Transaksi per Outlet dan per Tipe Penjualan

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

outlets = sorted(df_trx["outlet"].astype(str).unique())
ax[0].boxplot([df_trx.loc[df_trx["outlet"].astype(str) == o, "nilai_transaksi"] for o in outlets],
              labels=[o.replace("AYAM SERAYU - ", "") for o in outlets], showfliers=True,
              patch_artist=True,
              boxprops=dict(facecolor="#8ecae6"), medianprops=dict(color="#d00000", lw=2))
ax[0].set_title("Nilai Transaksi per Outlet")
ax[0].set_ylabel("Nilai transaksi (Rp)")

tipe = sorted(df_trx["tipe_penjualan"].astype(str).unique())
ax[1].boxplot([df_trx.loc[df_trx["tipe_penjualan"].astype(str) == t, "nilai_transaksi"] for t in tipe],
              labels=tipe, showfliers=True, patch_artist=True,
              boxprops=dict(facecolor="#ffb703"), medianprops=dict(color="#d00000", lw=2))
ax[1].set_title("Nilai Transaksi per Tipe Penjualan")
ax[1].set_ylabel("Nilai transaksi (Rp)")

plt.tight_layout()
simpan("02_boxplot_nilai_transaksi.png")
plt.show()

display(df_trx.groupby("outlet", observed=True)["nilai_transaksi"]
        .agg(["count", "median", "mean", "max"]).round(0))

## 8.3 Bar Chart — Pola Transaksi per Jam dan per Sesi

In [ ]:
per_jam = df_trx.groupby("jam").size()
per_sesi = df_trx["sesi"].value_counts().sort_index()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

warna_jam = ["#d00000" if j in [11, 12, 13, 18, 19] else "#219ebc" for j in per_jam.index]
ax[0].bar(per_jam.index, per_jam.values, color=warna_jam)
ax[0].set_title("Jumlah Transaksi per Jam (merah = jam sibuk)")
ax[0].set_xlabel("Jam"); ax[0].set_ylabel("Jumlah transaksi")
ax[0].set_xticks(range(0, 24, 2))

ax[1].barh(per_sesi.index.astype(str), per_sesi.values, color="#023047")
ax[1].set_title("Jumlah Transaksi per Sesi (hasil binning)")
ax[1].set_xlabel("Jumlah transaksi")
ax[1].invert_yaxis()

plt.tight_layout()
simpan("03_bar_pola_waktu.png")
plt.show()

## 8.4 Scatter Plot — Kuantitas vs Nilai Transaksi, Diwarnai Segmen

In [ ]:
sampel = df_trx.sample(n=min(8000, len(df_trx)), random_state=42)

plt.figure(figsize=(11, 5.5))
for seg in ["Kecil", "Sedang", "Besar"]:
    s = sampel[sampel["segmen_transaksi"] == seg]
    jitter = np.random.uniform(-0.22, 0.22, len(s))   # agar titik diskret tak saling menimpa
    plt.scatter(s["total_kuantitas"] + jitter, s["nilai_transaksi"],
                s=9, alpha=0.35, label=seg, color=WARNA[seg])

plt.axhline(Q1_TRX, color="#fb8500", ls="--", lw=1.5, label="Batas Q1")
plt.axhline(Q3_TRX, color="#d00000", ls="--", lw=1.5, label="Batas Q3")
plt.title("Hubungan Total Kuantitas dan Nilai Transaksi (sampel 8.000 transaksi)")
plt.xlabel("Total kuantitas item dalam transaksi")
plt.ylabel("Nilai transaksi (Rp)")
plt.legend(markerscale=2.5)
plt.tight_layout()
simpan("04_scatter_kuantitas_vs_nilai.png")
plt.show()

korelasi = df_trx[["total_kuantitas", "n_produk_unik", "nilai_transaksi",
                   "rata_harga_per_item"]].corr()
print("Matriks korelasi Pearson:")
display(korelasi.round(3))

## 8.5 Time Series — Tren Omzet Bulanan per Outlet

In [ ]:
bulanan = (df_trx.set_index("waktu")
                  .groupby("outlet", observed=True)["nilai_transaksi"]
                  .resample("MS").sum().reset_index())

plt.figure(figsize=(12, 5))
for o in sorted(bulanan["outlet"].astype(str).unique()):
    sub = bulanan[bulanan["outlet"].astype(str) == o]
    plt.plot(sub["waktu"], sub["nilai_transaksi"] / 1e6, marker="o", ms=3,
             label=o.replace("AYAM SERAYU - ", ""))

plt.title("Tren Omzet Bulanan per Outlet (2023–2025)")
plt.xlabel("Bulan"); plt.ylabel("Omzet (juta Rupiah)")
plt.legend(title="Outlet")
plt.tight_layout()
simpan("05_timeseries_omzet_bulanan.png")
plt.show()

print("Omzet total per tahun (juta Rupiah):")
display((df_trx.groupby(["tahun", "outlet"], observed=True)["nilai_transaksi"]
         .sum().unstack() / 1e6).round(1))

## 8.6 Bar Chart — Produk Terlaris berdasarkan Kuantitas dan Omzet

In [ ]:
per_produk = df.groupby("nama_produk", observed=True).agg(
    kuantitas=("jumlah_produk", "sum"),
    omzet=("penjualan_kotor", "sum"),
    kategori=("kategori", "first"),
).sort_values("omzet", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

top_qty = per_produk.sort_values("kuantitas", ascending=True)
warna_q = ["#ffb703" if k == "Minuman" else "#023047" for k in top_qty["kategori"]]
ax[0].barh(top_qty.index, top_qty["kuantitas"], color=warna_q)
ax[0].set_title("Kuantitas Terjual per Produk")
ax[0].set_xlabel("Jumlah unit terjual")

top_omz = per_produk.sort_values("omzet", ascending=True)
warna_o = ["#ffb703" if k == "Minuman" else "#023047" for k in top_omz["kategori"]]
ax[1].barh(top_omz.index, top_omz["omzet"] / 1e6, color=warna_o)
ax[1].set_title("Kontribusi Omzet per Produk")
ax[1].set_xlabel("Omzet (juta Rupiah)")

fig.suptitle("Biru tua = Makanan, Kuning = Minuman", y=1.01, fontsize=11)
plt.tight_layout()
simpan("06_bar_produk.png")
plt.show()

display(per_produk.assign(omzet_juta=(per_produk.omzet/1e6).round(1)).drop(columns="omzet"))

## 8.7 Stacked Bar — Komposisi Segmen di Tiap Sesi

In [ ]:
komposisi = pd.crosstab(df_trx["sesi"], df_trx["segmen_transaksi"], normalize="index") * 100

plt.figure(figsize=(11, 5))
bawah = np.zeros(len(komposisi))
for seg in ["Kecil", "Sedang", "Besar"]:
    plt.bar(komposisi.index.astype(str), komposisi[seg], bottom=bawah,
            label=seg, color=WARNA[seg])
    bawah += komposisi[seg].values

plt.title("Komposisi Segmen Transaksi di Tiap Sesi (%)")
plt.ylabel("Proporsi (%)"); plt.xlabel("Sesi")
plt.legend(title="Segmen", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=20)
plt.tight_layout()
simpan("07_stacked_segmen_per_sesi.png")
plt.show()

display(komposisi.round(2))

---
# LANGKAH 9 — Evaluasi dan Dokumentasi Akhir

## 9.1 Ringkasan Sebelum vs Sesudah Konstruksi

In [ ]:
ringkasan_akhir = pd.DataFrame([
    ("Jumlah baris (level item)", f"{len(df_raw):,}", f"{len(df):,}", "Tidak ada baris hilang"),
    ("Jumlah transaksi terdeteksi", f"{df_raw['ID Struk'].nunique():,}",
     f"{df_trx.shape[0]:,}", f"+{df_trx.shape[0] - df_raw['ID Struk'].nunique():,} transaksi dipulihkan"),
    ("Jumlah kolom", f"{df_raw.shape[1]}", f"{df.shape[1]} (item) / {df_trx.shape[1]} (transaksi)",
     "Fitur bertambah signifikan"),
    ("Total penjualan kotor", f"Rp{df_raw['Penjualan Kotor'].sum():,.0f}",
     f"Rp{df_trx['nilai_transaksi'].sum():,.0f}", "Identik — integritas terjaga"),
    ("Omzet bila dihitung naif", f"Rp{df_raw['Total'].sum():,.0f}",
     f"Rp{df_trx['nilai_transaksi'].sum():,.0f}",
     f"Koreksi {df_raw['Total'].sum()/df_trx['nilai_transaksi'].sum():.2f}x lipat"),
    ("Missing value", "0", "0", "Tidak ada imputasi diperlukan"),
    ("Kolom bervariansi nol", "3", "0", "status_pembayaran, diskon, pajak dibuang"),
    ("Kolom bertipe salah", "1 (waktu sebagai teks)", "0", "Sudah datetime64"),
    ("Label tersedia", "1 (kategori)", "3 (kategori, sesi, segmen_transaksi)", "2 label baru dengan SOP"),
], columns=["aspek", "sebelum", "sesudah", "keterangan"])

pd.set_option("display.max_colwidth", 70)
ringkasan_akhir.set_index("aspek")

## 9.2 Penilaian Kualitas Data (Sebelum vs Sesudah)

In [ ]:
dimensi_mutu = pd.DataFrame([
    ("Kelengkapan (Completeness)", 100, 100, "Tidak ada missing value sejak awal"),
    ("Keunikan (Uniqueness)", 98.8, 100.0, "Tabrakan ID Struk teratasi lewat kunci komposit"),
    ("Konsistensi (Consistency)", 24.0, 100.0, "Metode pembayaran dinaikkan ke level transaksi"),
    ("Keakuratan (Accuracy)", 100, 100, "Aturan Jumlah x Harga = Kotor lolos 100% sejak awal"),
    ("Validitas (Validity)", 100, 100, "Tidak ada nilai di luar domain yang wajar"),
    ("Kesiapan analitis (Fitness)", 40, 95, "Granularitas benar + fitur & label tersedia"),
], columns=["dimensi", "skor_sebelum_%", "skor_sesudah_%", "dasar penilaian"])
dimensi_mutu["perubahan"] = (dimensi_mutu["skor_sesudah_%"] - dimensi_mutu["skor_sebelum_%"]).round(1)
display(dimensi_mutu.set_index("dimensi"))

print("Catatan: skor Konsistensi 'sebelum' = proporsi transaksi yang metode pembayarannya")
print("seragam antar-baris (24,0%), dihitung langsung dari data di sel 1.5.5.")

## 9.3 Kesimpulan

1. **Data yang tampak bersih belum tentu bersih.** Dataset ini tidak memiliki satu pun missing value,
   duplikat yang perlu dihapus, maupun nilai di luar rentang wajar. Namun ia menyimpan dua cacat
   struktural serius — tabrakan ID dan salah granularitas — yang tidak akan terdeteksi oleh
   `df.info()` maupun `df.describe()`. Yang menemukannya adalah **validasi aturan bisnis**.

2. **Keputusan pembersihan harus diuji, bukan diasumsikan.** Tindakan refleks `drop_duplicates()`
   terbukti akan merusak data: uji rekonsiliasi menunjukkan 10.131 transaksi kehilangan kecocokan
   nilai bila baris duplikat dihapus. Duplikat tersebut ternyata pencatatan yang sah.

3. **Granularitas menentukan kebenaran angka.** Menghitung omzet tanpa agregasi menghasilkan nilai
   sekitar 3,6 kali lipat dari yang sebenarnya. Semua metrik nilai dalam notebook ini dihitung pada
   level transaksi.

4. **Pelabelan menuntut SOP dan kesadaran terhadap kebocoran.** Label `segmen_transaksi` diturunkan
   dari `nilai_transaksi`, sehingga daftar kolom terlarang disusun secara eksplisit agar tidak
   menghasilkan model yang akurat semu.

5. **Hasil akhir:** dari 15 kolom mentah bergranularitas ambigu menjadi dua tabel siap analisis
   (626.311 baris item dan 208.730 transaksi) dengan lebih dari 40 fitur, tiga label bervalidasi,
   dan nol baris data yang hilang.

## 9.4 Rekomendasi Tindak Lanjut

**Untuk kualitas data (sisi sumber / sistem POS)**

| Prioritas | Tindakan | Dampak yang diharapkan |
|---|---|---|
| Tinggi | Ubah skema ID struk menjadi unik global | Menghilangkan tabrakan ID di sumbernya |
| Tinggi | Pindahkan metode pembayaran ke tabel header transaksi | Konsistensi naik dari 24,0% ke 100% |
| Sedang | Ekspor dalam dua tabel (header & detail) | Granularitas tidak lagi ambigu |
| Sedang | Rekam status pembayaran sebenarnya (void, refund) | Analisis kebocoran pendapatan jadi mungkin |
| Rendah | Tambahkan identitas pelanggan | Membuka analisis RFM, retensi, dan CLV |

**Untuk analisis lanjutan (memakai keluaran notebook ini)**

1. **Market basket analysis** memakai `df` level item — mencari produk yang sering dibeli bersamaan
   sebagai dasar penyusunan paket bundling.
2. **Klasifikasi `segmen_transaksi`** memakai `FITUR_AMAN` — memprediksi apakah sebuah transaksi akan
   bernilai besar berdasarkan konteks waktu dan komposisi pesanan.
3. **Peramalan permintaan** memakai agregat harian per outlet — mendukung perencanaan stok dan
   penjadwalan shift kasir.
4. **Analisis attachment rate minuman** memakai `punya_minuman` dan `rasio_nilai_minuman` — mengukur
   efektivitas upselling per sesi dan per outlet.

> **Peringatan yang wajib dibawa ke analisis lanjutan:** kolom `metode_pembayaran` memiliki
> keterbatasan yang dijelaskan di Langkah 1.5.5 dan 5.3. Setiap temuan yang berbasis kolom tersebut
> harus mencantumkan disclaimer ini.

## 9.5 Ekspor Seluruh Hasil

In [ ]:
import json

# 1. Dataset level transaksi
PATH_TRX = os.path.join(OUTPUT_DIR, "03_data_level_transaksi.csv")
df_trx.to_csv(PATH_TRX, index=False)

# 2. Dataset siap model (fitur aman + label)
df_model = df_trx[["id_transaksi"] + FITUR_AMAN + ["segmen_transaksi"]].copy()
PATH_MODEL = os.path.join(OUTPUT_DIR, "04_dataset_siap_model.csv")
df_model.to_csv(PATH_MODEL, index=False)

# 3. Artefak SOP pelabelan
PATH_SOP = os.path.join(OUTPUT_DIR, "05_artefak_sop_pelabelan.json")
artefak = {
    "label": "segmen_transaksi",
    "level": "transaksi",
    "batas": {"Q1": Q1_TRX, "Q3": Q3_TRX},
    "aturan": {"Kecil": f"nilai_transaksi <= {Q1_TRX:.0f}",
               "Sedang": f"{Q1_TRX:.0f} < nilai_transaksi <= {Q3_TRX:.0f}",
               "Besar": f"nilai_transaksi > {Q3_TRX:.0f}"},
    "batas_sesi": dict(zip(LABEL_SESI, ["00-05", "06-10", "11-14", "15-17", "18-21", "22-23"])),
    "kolom_bocor": KOLOM_BOCOR,
    "fitur_aman": FITUR_AMAN,
    "n_transaksi_acuan": int(len(df_trx)),
}
with open(PATH_SOP, "w", encoding="utf-8") as f:
    json.dump(artefak, f, indent=2, ensure_ascii=False)

# 4. Laporan ringkas
PATH_LAPORAN = os.path.join(OUTPUT_DIR, "06_laporan_ringkas.csv")
ringkasan_akhir.to_csv(PATH_LAPORAN, index=False)

daftar = []
for akar, _, berkas in os.walk(OUTPUT_DIR):
    for b in sorted(berkas):
        p = os.path.join(akar, b)
        daftar.append({"file": os.path.relpath(p, OUTPUT_DIR),
                       "ukuran_MB": round(os.path.getsize(p) / 1024**2, 2)})

print("SELURUH BERKAS KELUARAN\n")
display(pd.DataFrame(daftar))
print(f"\nDataset siap model: {df_model.shape[0]:,} baris x {df_model.shape[1]} kolom")
print("Notebook selesai dijalankan tanpa error.")

---

## Penutup

Seluruh Langkah 1–9 sesuai instruksi tugas telah dikerjakan. Berkas keluaran di folder `output/`:

| Berkas | Isi |
|---|---|
| `01_data_bersih_level_item.csv` | Data bersih granularitas baris item |
| `02_kamus_data_fitur.csv` | Kamus data seluruh fitur hasil konstruksi |
| `03_data_level_transaksi.csv` | Data agregat level transaksi + fitur + label |
| `04_dataset_siap_model.csv` | Fitur bebas kebocoran + label target |
| `05_artefak_sop_pelabelan.json` | Batas kelas & aturan SOP untuk dipakai ulang |
| `06_laporan_ringkas.csv` | Tabel ringkasan sebelum–sesudah |
| `figures/*.png` | Tujuh visualisasi |